In [1]:
"""Schrödinger equation"""

__author__    = "Evgueni Dinvay"
__credit__    = ["Evgueni Dinvay", "Valentyn Bezshapkin"]

__date__      = "2026-08-13"

# The Schrödinger equation



Here we demonstrate how the multiwavelets machinery can be exploited for time evolution simulations
of the following one dimensional equation
\begin{equation}
    i \partial_t \Psi
    =
    \left( - \partial_x^2 + V(x) \right) \Psi
    ,
\end{equation}
where we restrict ourselves to consideration of a time independent potential $V$.
This equation is complemented by a given initial wave function $\Psi(x, 0) = \Psi_0(x)$.
As an example, we take a harmonic potential and a gaussian initial wave packet.


A common way of numerical treatment of this equation
is to choose a small time step $t > 0$ and construct the time evolution operator
\begin{equation}
    \Psi(t)
    =
    \exp
    \left(
        i t \partial_x^2 -i t V
    \right)
    \Psi_0
\end{equation}
on interval $[0, t]$.
Then applying it iteratively one can arrive to the solution
at some given finite time moment,
thanks to the semigroup property of the propagator.
As we normally do not have an access to the full collection of eigenfunctions and eigenvalues
for the Hamiltonian, we have to split the propagator on kinetic
$
    \exp
    \left(
        i t \partial_x^2
    \right)
$
and potential 
$
    \exp
    \left(
        -i t V
    \right)
$
parts.
Note that the latter is a multiplication operator in the physical space.


We will make use of the following fourth order scheme
\begin{equation}
    \exp ( At + Bt )
    =
    \exp \left(  \frac t6  B \right)
    \exp \left(  \frac t2  A \right)
    \exp \left(  \frac {2t}3  \widetilde B \right)
    \exp \left(  \frac t2  A \right)
    \exp \left(  \frac t6  B \right)
    +
    \mathcal O \left( t^5 \right)
    ,
\end{equation}
where
\begin{equation}
    \widetilde B
    =
    B
    +
    \frac{t^2}{48}
    [ B, [A, B]]
    .
\end{equation}
In the case of $A = i \partial_x^2$ and $B = -i V$
this
$
    \widetilde B
$
turns out to be a multiplication operator containing potential gradient $\partial_x V(x)$.
Remarkably,
this high order scheme requires only two applications
of the free-particle semigroup operator
$
    \exp \left(  it  \partial_x^2 / 2 \right)
$
per time step.

In [2]:
class ChinChenA(object):
    def __init__(self, expA, expB, exp_tildeB):
        self.expA = expA
        self.expB = expB
        self.exp_tildeB = exp_tildeB
    def __call__(self, u):
        u = self.expB(u)        # 1/6*dt
        u = self.expA(u)        # 1/2*dt
        u = self.exp_tildeB(u)  # 2/3*dt
        u = self.expA(u)        # 1/2*dt
        u = self.expB(u)        # 1/6*dt
        return u    

`expA`: stands for operator $\exp \left(  \frac t2  A \right) = \exp \left(  i \frac t2  \partial_x^2 \right)$


`expB`: stands for operator $\exp \left(  \frac t6  B \right) = \exp \left(  -i \frac t6  V \right)$


`exp_tildeB`: stands for operator
$
    \exp \left(  \frac {2t}3  \widetilde B \right) = \exp \left(  -i \frac {2t}3  \widetilde V \right)
$
with
$
    \widetilde V
    =
    V
    -
    \frac{t^2}{24}
    ( \partial_x V )^2
$

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from vampyr import vampyr1d as vp1
from vampyr import LegendreBasis

# Define parameters, final time moment and time step
x0 = 0.3
sigma = 0.04
x1 = 0.5
V0 = 25000
N = 10
t_period = np.pi / np.sqrt(V0)
time_step = 0.5 * t_period / N

# Set the precision and make the MRA
precision = 1.0e-5
finest_scale = 9
mra = vp1.MultiResolutionAnalysis(vp1.BoundingBox(0), LegendreBasis(5))

# Make the scaling projectors. The complex one produces ComplexFunctionTrees
# from complex-valued callables; the real one is kept for the initial state.
P = vp1.ScalingProjector(mra, prec = precision)
Pc = vp1.ScalingProjector(mra, prec = precision, dtype = complex)

## Representing the wave function

The wave function is complex, and so are both factors of the splitting. VAMPyR
represents such a function as a single `ComplexFunctionTree`, so
$\Psi$ is one object rather than a pair $(\Re \Psi, \Im \Psi)$, and an
operator acting on it is an ordinary callable rather than a $2 \times 2$ real
block

\begin{equation*}
    U
    =
    \begin{pmatrix}
        \Re U & - \Im U \\
        \Im U & \Re U
    \end{pmatrix}
    .
\end{equation*}

The block form is what one is forced into when only real trees are available;
it is worth keeping in mind as the structure underneath, but none of it has to
be written out below.

The unitary potential semigroup

\begin{equation*}
    U \left( V, t \right) = \exp \left( -i V t \right)
\end{equation*}

is a multiplication operator, so it is enough to project the complex phase
$\exp(-iVt)$ once and multiply by it.

In [4]:
class PotentialPropagator(object):
    """exp(-i V t), as multiplication by a projected complex phase."""
    def __init__(self, Pc, V, t):
        self.phase = Pc(lambda x: np.exp(-1j * V(x) * t))
    def __call__(self, psi):
        return self.phase * psi

The Schrodinger free-particle semigroup

\begin{equation*}
    U \left( \widehat T, t \right)
    =
    U \left( - \partial_x^2, t \right)
    =
    \exp \left( it \partial_x^2 \right)
\end{equation*}

is a genuinely complex convolution: its kernel carries both a $\cos$ and a
$\sin$ part. `ComplexTimeEvolutionOperator` builds it as one operator with one
complex kernel, so a single application does the whole propagation -- where
previously two real operators had to be constructed and their results combined
as $\left( \Re U \right) \Psi + i \left( \Im U \right) \Psi$.

It is built at a fixed finest scale; there is no adaptive constructor on the
complex side.

It is well known that in the harmonic potential
$
    V(x) = V_0 (x - x_1)^2
$
the density
$
    \left| \Psi(t) \right|^2
$
oscillates with the period
$
    t_{\text{period}} = \pi / \sqrt{V_0}
    .
$
More precisely,
$
    \Psi \left( t_{\text{period}} \right) = - \Psi_0
    .
$
This can immediately be seen taking into account that
the eigenvalues for the Hamiltonian are
$
    \sqrt{V_0} ( 2n + 1)
    .
$

For the time step $t$ the scheme modification potential
$
    \widetilde V
    =
    V
    -
    \frac{t^2}{24}
    ( \partial_x V )^2
    =
    \widetilde V_0 (x - x_0)^2
$
where
$
    \widetilde V_0
    =
    V_0
    -
    \frac{(tV_0)^2}6
    .
$

In [ ]:
# Define the harmonic potential with its scheme modification
def V(x):
    return V0 * (x[0] - x1)**2
def tilde_V(x):
    A = V0 - ( time_step * V0 )**2 / 6.0
    return A * (x[0] - x1)**2

# Define the iteration procedure
iteratorA = ChinChenA(
    expA = vp1.TimeEvolutionOperator(mra, precision, time_step / 2, finest_scale),
    expB = PotentialPropagator(Pc, V, time_step / 6),
    exp_tildeB = PotentialPropagator(Pc, tilde_V, 2 * time_step / 3)
)

AttributeError: module 'vampyr._vampyr.vampyr1d' has no attribute 'ComplexTimeEvolutionOperator'

Let an initial condition be of the form
\begin{equation*}
    \Psi_0(x)
    =
    \left(
        \frac 1{2 \pi \sigma^2}
    \right)^{1/4}
    \exp
    \left(
        - \frac {(x - x_0)^2}{4 \sigma^2}
    \right)
\end{equation*}
with $x_0 = 0.3$ and $\sigma = 0.04$.
All the parameters are chosen in a way that the solution
stays localized mainly on the space interval $[0, 1]$.

In [ ]:
# Define the initial wave function. It is real, but projected straight into a
# complex tree -- there is no zero imaginary part to carry alongside.
psi0 = vp1.GaussFunc(
    beta = 1.0 / (4 * sigma**2), alpha = (2 * np.pi * sigma**2)**(-1/4), position = [x0]
)
psi0 = P(psi0).to_complex()

# Plot the initial density
x_vec = np.linspace(0, 0.9999, 1000)
density0 = (psi0 * psi0.conj()).real()
plt.plot(x_vec, [density0([x]) for x in x_vec], '--', color='black')

# Solve the initial value problem and plot the result
psiA = psi0
for n in range(N):
    psiA = iteratorA(psiA)
    densityA = (psiA * psiA.conj()).real()
    plt.plot(x_vec, [densityA([x]) for x in x_vec], "black")

plt.xlabel('x')
plt.ylabel(r'$| \Psi(x, t) |^2$')
plt.title(r'Density time evolution')
plt.show()

In [ ]:
# Continue solving until t = period
for n in range(N):
    psiA = iteratorA(psiA)

# Find error at t = period
per_errorA = psiA + psi0

print( f"L2-norm of the error:            {per_errorA.norm()}" )
print( f"L2-norm of real part error:      {per_errorA.real().norm()}" )
print( f"L2-norm of imaginary part error: {per_errorA.imag().norm()}" )

plt.plot(x_vec, [per_errorA.real()([x]) for x in x_vec], "blue",
         label=r'Re$( \Psi(x, t_{ \text{period} }) + \Psi_0(x) )$')
plt.plot(x_vec, [per_errorA.imag()([x]) for x in x_vec], "orange",
         label=r'Im$( \Psi(x, t_{ \text{period} }) + \Psi_0(x) )$')

plt.xlabel('x')
plt.title(r'Difference with the exact solution at $t = t_{ \text{period} }$')
plt.legend()
plt.show()